In [ ]:
!pip install requests beautifulsoup4 pandas lxml -q
print('Listo!')

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from urllib.parse import quote
import random

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0',
    'Accept': 'text/html',
    'Accept-Language': 'en-US,en;q=0.9',
}
session = requests.Session()
session.headers.update(headers)

def extraer_apellido(nombre):
    if pd.isna(nombre):
        return None
    nombre = str(nombre).strip()
    apellido = re.sub(r'^([A-Z]\.\s*)+', '', nombre).strip()
    return apellido if apellido else nombre

def normalizar(texto):
    if not texto:
        return ''
    texto = texto.lower()
    for k, v in {'\u00e1':'a','\u00e9':'e','\u00ed':'i','\u00f3':'o','\u00fa':'u','\u00f1':'n'}.items():
        texto = texto.replace(k, v)
    return texto

def buscar_jugador(nombre_original, equipo=None):
    try:
        apellido = extraer_apellido(nombre_original)
        if not apellido:
            return None, None
        url = 'https://www.transfermarkt.com/schnellsuche/ergebnis/schnellsuche?query=' + quote(apellido)
        r = session.get(url, timeout=15)
        if r.status_code != 200:
            return None, None
        soup = BeautifulSoup(r.content, 'lxml')
        tables = soup.find_all('table', class_='items')
        if not tables:
            return None, None
        rows = tables[0].find_all('tr', class_=['odd', 'even'])
        candidatos = []
        for row in rows:
            for link in row.find_all('a'):
                href = link.get('href', '')
                if '/profil/spieler/' in href:
                    player_url = 'https://www.transfermarkt.com' + href
                    player_name = link.get_text(strip=True)
                    score = 0
                    row_text = normalizar(row.get_text())
                    if equipo:
                        for palabra in normalizar(equipo).split():
                            if len(palabra) > 3 and palabra in row_text:
                                score += 2
                                break
                    candidatos.append({'url': player_url, 'nombre': player_name, 'score': score})
                    break
        if not candidatos:
            return None, None
        candidatos.sort(key=lambda x: x['score'], reverse=True)
        return candidatos[0]['url'], candidatos[0]['nombre']
    except:
        return None, None

def obtener_datos(url):
    datos = {'transfermarkt_url': url, 'nombre_tm': None, 'valor_mercado': None, 'agente': None, 'fin_contrato': None, 'imagen_url': None}
    try:
        r = session.get(url, timeout=15)
        if r.status_code != 200:
            return datos
        soup = BeautifulSoup(r.content, 'lxml')
        h1 = soup.find('h1', class_='data-header__headline-wrapper')
        if h1:
            datos['nombre_tm'] = h1.get_text(strip=True)
        mv = soup.find('a', class_='data-header__market-value-wrapper')
        if mv:
            datos['valor_mercado'] = mv.get_text(strip=True)
        img = soup.find('img', class_='data-header__profile-image')
        if img:
            datos['imagen_url'] = img.get('src')
        agent_link = soup.find('a', href=re.compile(r'/berater/'))
        if agent_link:
            datos['agente'] = agent_link.get_text(strip=True)
        contract = re.search(r'Contract expires:\s*(\w+\s+\d+,\s+\d{4})', soup.get_text())
        if contract:
            datos['fin_contrato'] = contract.group(1)
        return datos
    except:
        return datos

print('Funciones cargadas!')

In [ ]:
# Cargar CSV
csv_base = 'https://docs.google.com/spreadsheets/d/e/'
csv_id = '2PACX-1vSneBjGlw2I3SyXV-uw1V8Cs_O4lbiQw39melKEZJNhunpshakPrn7AZQBN2L8N9Yw_HA-EeVOt3qvf'
csv_params = '/pub?gid=2002620668&single=true&output=csv'
CSV_URL = csv_base + csv_id + csv_params

df = pd.read_csv(CSV_URL)
print(f'Jugadores cargados: {len(df)}')
df[['Jugador', 'Equipo', 'Liga']].head(10)

In [ ]:
# CONFIGURACION - Modificar aqui
DESDE = 0      # Fila inicial
HASTA = 30     # Fila final (usa len(df) para todos)
PAUSA_MIN = 3  # Segundos minimo entre requests
PAUSA_MAX = 5  # Segundos maximo

total = min(HASTA, len(df)) - DESDE
print(f'Procesara {total} jugadores')
print(f'Tiempo estimado: {total * 4 / 60:.1f} minutos')

In [ ]:
# EJECUTAR SCRAPING
resultados = []
total = min(HASTA, len(df)) - DESDE

print(f'Procesando {total} jugadores...\n')

for idx in range(DESDE, min(HASTA, len(df))):
    row = df.iloc[idx]
    nombre = row['Jugador']
    equipo = row.get('Equipo', '')
    liga = row.get('Liga', '')
    
    print(f'[{idx+1-DESDE}/{total}] {nombre} ({equipo})...', end=' ')
    
    url, nombre_tm = buscar_jugador(nombre, equipo)
    
    if url:
        print(f'OK -> {nombre_tm}')
        datos = obtener_datos(url)
        datos['jugador_csv'] = nombre
        datos['equipo_csv'] = equipo
        datos['liga_csv'] = liga
        resultados.append(datos)
    else:
        print('NO ENCONTRADO')
        resultados.append({
            'jugador_csv': nombre,
            'equipo_csv': equipo,
            'liga_csv': liga,
            'transfermarkt_url': None,
            'nombre_tm': None,
            'valor_mercado': None,
            'agente': None,
            'fin_contrato': None,
            'imagen_url': None
        })
    
    time.sleep(random.uniform(PAUSA_MIN, PAUSA_MAX))

print(f'\n=== LISTO ===')
encontrados = sum(1 for r in resultados if r['transfermarkt_url'])
print(f'Encontrados: {encontrados}/{total}')

In [ ]:
# Ver resultados
df_result = pd.DataFrame(resultados)
df_result[['jugador_csv', 'nombre_tm', 'equipo_csv', 'valor_mercado']].head(20)

In [ ]:
# Guardar y descargar
archivo = f'transfermarkt_{DESDE}_{HASTA}.csv'
df_result.to_csv(archivo, index=False, encoding='utf-8-sig')
print(f'Guardado: {archivo}')

from google.colab import files
files.download(archivo)

In [ ]:
# Ver no encontrados
no_encontrados = df_result[df_result['transfermarkt_url'].isna()]
print(f'No encontrados: {len(no_encontrados)}')
for _, r in no_encontrados.iterrows():
    print(f"  - {r['jugador_csv']} ({r['equipo_csv']})")